# Part 1 - Modelagem com tracking via MLflow

Este notebook treina uma primeira versao do detector de `carpet` da Part 1 usando FFT/PSD e XGBoost.

In [ ]:
from pathlib import Path
import json
import tempfile
import warnings

import joblib
import mlflow
import mlflow.xgboost
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import signal
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 120)

PROJECT_CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next(
    (
        path
        for path in PROJECT_CANDIDATES
        if (path / 'source' / 'part1' / 'labels.csv').exists()
        or (path / 'data' / 'part1' / 'labels.csv').exists()
    ),
    Path.cwd().resolve(),
)

PART1_CANDIDATES = [
    PROJECT_ROOT / 'source' / 'part1',
    PROJECT_ROOT / 'data' / 'part1',
]
PART1_DIR = next(
    (path for path in PART1_CANDIDATES if (path / 'labels.csv').exists() and (path / 'data').exists()),
    None,
)
if PART1_DIR is None:
    caminhos = '\n'.join(str(path) for path in PART1_CANDIDATES)
    raise FileNotFoundError(f'Nao encontrei os dados da part1. Caminhos testados:\n{caminhos}')

DATA_DIR = PART1_DIR / 'data'
MODEL_NAME = 'Modelo_mlflow_v1s'
MODEL_DIR = PROJECT_ROOT / 'models' / 'part1'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PKL_PATH = MODEL_DIR / f'{MODEL_NAME}.pkl'

MLFLOW_TRACKING_DIR = PROJECT_ROOT / 'mlruns'
mlflow.set_tracking_uri(MLFLOW_TRACKING_DIR.resolve().as_uri())
mlflow.set_experiment('part1_modelagem_mlflow_v1s')

#parâmetros iniciais do modelo
RANDOM_STATE = 42
MIN_FREQ_HZ = 1000
WINDOW_HZ = 250
STEP_HZ = 125
MIN_OVERLAP_FRACTION = 0.20
EPS = 1e-18

print(f'Projeto: {PROJECT_ROOT}')
print(f'Dados Part 1: {PART1_DIR}')
print(f'MLflow tracking URI: {mlflow.get_tracking_uri()}')
print(f'Modelo final sera salvo em: {MODEL_PKL_PATH}')

c:\Users\vinicius\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\utils\requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251
2026/07/28 22:51:34 INFO mlflow.tracking.fluent: Experiment with name 'part1_modelagem_mlflow_v1s' does not exist. Creating a new experiment.


Projeto: C:\Users\vinicius\Documents\MEUGITHUB\Vibration_challenge
Dados Part 1: C:\Users\vinicius\Documents\MEUGITHUB\Vibration_challenge\source\part1
MLflow tracking URI: file:///C:/Users/vinicius/Documents/MEUGITHUB/Vibration_challenge/mlruns
Modelo final sera salvo em: C:\Users\vinicius\Documents\MEUGITHUB\Vibration_challenge\models\part1\Modelo_mlflow_v1s.pkl


In [2]:
# Leitura dos labels
labels = pd.read_csv(PART1_DIR / 'labels.csv')
labels['sample_id'] = labels['sample_id'].astype(str)
labels['regions_obj'] = labels['regions'].apply(json.loads)
labels['tem_carpet'] = labels['regions_obj'].apply(lambda regions: len(regions) > 0)
labels['n_regioes'] = labels['regions_obj'].apply(len)

display(labels.head())

,sample_id,regions,regions_obj,tem_carpet,n_regioes
0,ef823432-8249-5cba-ace1-3c5192b825b7,"[[4000.0, 5300.0]]","[[4000.0, 5300.0]]",True,1
1,9b7308b8-1db3-5130-b498-b81dffaedd7d,"[[4100.0, 5750.0]]","[[4100.0, 5750.0]]",True,1
2,e5b6beb7-fbd1-5bfb-ae40-fb61922f2215,"[[3375.0, 3643.0], [4000.0, 5250.0]]","[[3375.0, 3643.0], [4000.0, 5250.0]]",True,2
3,0209af20-34f8-5c0e-bdad-94b1e7046f71,"[[3525.0, 4438.0]]","[[3525.0, 4438.0]]",True,1
4,505894c3-1d24-560f-9588-d758b1fb7004,"[[3760.0, 4000.0], [4800.0, 5640.0]]","[[3760.0, 4000.0], [4800.0, 5640.0]]",True,2


In [4]:
# distribuição de casos positivos e negativos para carpet
display(labels['tem_carpet'].value_counts().rename(index={False: 'sem_carpet', True: 'com_carpet'}))

tem_carpet
com_carpet    100
sem_carpet     26
Name: count, dtype: int64

In [6]:
# Funções auxiliares para leitura do sinal e calculo do PSD

def taxa_amostragem(tempo):
    dt = np.diff(np.asarray(tempo, dtype=np.float64))
    dt = dt[dt > 0]
    if len(dt) == 0:
        return np.nan
    return 1.0 / np.median(dt)


def carregar_sinal(sample_id):
    caminho = DATA_DIR / f'{sample_id}.csv'
    df = pd.read_csv(caminho)
    t_col = 't' if 't' in df.columns else df.columns[0]
    data_col = 'data' if 'data' in df.columns else df.columns[-1]
    tempo = df[t_col].to_numpy(dtype=np.float64)
    sinal = df[data_col].to_numpy(dtype=np.float64)
    fs = taxa_amostragem(tempo)
    return tempo, sinal, fs


def calcular_psd(tempo, sinal, nperseg_max=8192):
    sinal = np.asarray(sinal, dtype=np.float64)
    sinal = signal.detrend(sinal - np.mean(sinal))
    fs = taxa_amostragem(tempo)
    nperseg = min(nperseg_max, len(sinal))

    frequencia, psd = signal.welch(
        sinal,
        fs=fs,
        window='hann',
        nperseg=nperseg,
        noverlap=nperseg // 2,
        scaling='density',
    )
    return frequencia, psd


def calcular_fft(tempo, sinal):
    sinal = np.asarray(sinal, dtype=np.float64)
    sinal = signal.detrend(sinal - np.mean(sinal))
    fs = taxa_amostragem(tempo)
    janela = signal.windows.hann(len(sinal), sym=False)
    fft_valores = np.fft.rfft(sinal * janela)
    frequencia = np.fft.rfftfreq(len(sinal), d=1.0 / fs)
    amplitude = np.abs(fft_valores) * 2.0 / np.sum(janela)
    amplitude_db = 20 * np.log10(amplitude + 1e-12)
    return frequencia, amplitude_db

## Criação de features para enriquecimento do modelo data driven

Calculamos o PSD na sequência temos o filtro da região de 1K Hz. 

Criamos um janelamento de 250Hz com o deslocamento de 125Hz. 

Para cada trecho desse janelamento é calculado algumas informações estatisticas do mesmo e marcamos com o label se essa é uma região de carpet ou não.

In [7]:
def energia_banda(freq, psd, inicio_hz, fim_hz):
    mask = (freq >= inicio_hz) & (freq < fim_hz)
    if mask.sum() < 2:
        return 0.0
    return float(np.trapz(psd[mask], freq[mask]))


def spectral_flatness(valores):
    valores = np.asarray(valores, dtype=np.float64)
    valores = np.maximum(valores, EPS)
    return float(np.exp(np.mean(np.log(valores))) / (np.mean(valores) + EPS))


def overlap_com_regioes(inicio_hz, fim_hz, regioes):
    total = 0.0
    for reg_inicio, reg_fim in regioes:
        total += max(0.0, min(fim_hz, reg_fim) - max(inicio_hz, reg_inicio))
    return total


def features_de_janela(sample_id, tempo, sinal, regioes):
    freq, psd = calcular_psd(tempo, sinal)
    log_psd = 10 * np.log10(psd + EPS)

    energia_total_acima_1k = energia_banda(freq, psd, MIN_FREQ_HZ, freq.max()) + EPS
    freq_max = float(freq.max())
    linhas = []

    inicio = MIN_FREQ_HZ
    while inicio + WINDOW_HZ <= freq_max:
        fim = inicio + WINDOW_HZ
        mask = (freq >= inicio) & (freq < fim)
        if mask.sum() >= 3:
            freq_j = freq[mask]
            psd_j = psd[mask]
            log_j = log_psd[mask]

            overlap_hz = overlap_com_regioes(inicio, fim, regioes)
            overlap_fraction = overlap_hz / WINDOW_HZ
            alvo = int(overlap_fraction >= MIN_OVERLAP_FRACTION)

            peaks, _ = signal.find_peaks(log_j, prominence=0.5)
            slope = np.polyfit(freq_j, log_j, 1)[0] if len(freq_j) >= 2 else 0.0

            linhas.append(
                {
                    'sample_id': sample_id,
                    'start_hz': inicio,
                    'end_hz': fim,
                    'center_hz': inicio + WINDOW_HZ / 2,
                    'overlap_hz': overlap_hz,
                    'overlap_fraction': overlap_fraction,
                    'target': alvo,
                    'energia_relativa_janela': energia_banda(freq, psd, inicio, fim) / energia_total_acima_1k,
                    'log_psd_mean': float(np.mean(log_j)),
                    'log_psd_std': float(np.std(log_j)),
                    'log_psd_max': float(np.max(log_j)),
                    'log_psd_p90': float(np.quantile(log_j, 0.90)),
                    'log_psd_p10': float(np.quantile(log_j, 0.10)),
                    'log_psd_range': float(np.max(log_j) - np.min(log_j)),
                    'slope_log_psd': float(slope),
                    'flatness_psd': spectral_flatness(psd_j),
                    'n_picos_log': int(len(peaks)),
                    'densidade_picos': float(len(peaks) / (WINDOW_HZ / 1000)),
                    'std_sinal': float(np.std(sinal)),
                    'max_abs_sinal': float(np.max(np.abs(sinal))),
                }
            )

        inicio += STEP_HZ

    return linhas


def criar_dataset(labels_df):
    todas_linhas = []
    for row in labels_df.itertuples(index=False):
        tempo, sinal, _ = carregar_sinal(row.sample_id)
        todas_linhas.extend(features_de_janela(row.sample_id, tempo, sinal, row.regions_obj))
    return pd.DataFrame(todas_linhas)


dataset = criar_dataset(labels)
print(f'Total de janelas criadas: {len(dataset):,}')

Total de janelas criadas: 6,752


In [8]:
display(dataset.head())

,sample_id,start_hz,end_hz,center_hz,overlap_hz,overlap_fraction,target,energia_relativa_janela,log_psd_mean,log_psd_std,log_psd_max,log_psd_p90,log_psd_p10,log_psd_range,slope_log_psd,flatness_psd,n_picos_log,densidade_picos,std_sinal,max_abs_sinal
0,ef823432-8249-5cba-ace1-3c5192b825b7,1000,1250,1125.0,0.0,0.0,0,1.768891e-14,-169.200212,1.581182,-165.858344,-167.074692,-171.092721,9.540720,-0.007466,8.536772e-01,28,112.0,0.418864,2.381361
1,ef823432-8249-5cba-ace1-3c5192b825b7,1125,1375,1250.0,0.0,0.0,0,2.007080e-03,-166.607083,19.000560,-39.377299,-167.484148,-171.283394,136.021766,0.035332,1.485691e-11,29,116.0,0.418864,2.381361
2,ef823432-8249-5cba-ace1-3c5192b825b7,1250,1500,1375.0,0.0,0.0,0,2.007080e-03,-166.669721,19.007064,-39.377299,-167.794392,-171.617242,134.945790,-0.036838,1.462964e-11,28,112.0,0.418864,2.381361
3,ef823432-8249-5cba-ace1-3c5192b825b7,1375,1625,1500.0,0.0,0.0,0,1.556217e-14,-169.694361,1.539145,-166.378291,-167.948816,-171.653176,7.944797,-0.000365,8.480613e-01,28,112.0,0.418864,2.381361
4,ef823432-8249-5cba-ace1-3c5192b825b7,1500,1750,1625.0,0.0,0.0,0,1.616580e-14,-169.560850,1.601720,-165.267463,-167.642838,-171.568895,8.778267,0.003082,8.436373e-01,31,124.0,0.418864,2.381361


In [9]:
display(dataset['target'].value_counts().rename(index={0: 'sem_carpet', 1: 'com_carpet'}))

target
sem_carpet    5327
com_carpet    1425
Name: count, dtype: int64

### Split de dados de treino e de teste. 

Vale ressaltar que devemos fazer esse split mas considerar que devem ser ID destintos para evitar "aprender padrões específicos"


In [10]:
sample_table = labels[['sample_id', 'tem_carpet']].copy()

train_ids, temp_ids = train_test_split(
    sample_table['sample_id'],
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=sample_table['tem_carpet'],
)

temp_table = sample_table[sample_table['sample_id'].isin(temp_ids)]
val_ids, test_ids = train_test_split(
    temp_table['sample_id'],
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_table['tem_carpet'],
)

train_ids = set(train_ids)
val_ids = set(val_ids)
test_ids = set(test_ids)

train_df = dataset[dataset['sample_id'].isin(train_ids)].reset_index(drop=True)
val_df = dataset[dataset['sample_id'].isin(val_ids)].reset_index(drop=True)
test_df = dataset[dataset['sample_id'].isin(test_ids)].reset_index(drop=True)

FEATURE_COLUMNS = [
    'center_hz',
    'energia_relativa_janela',
    'log_psd_mean',
    'log_psd_std',
    'log_psd_max',
    'log_psd_p90',
    'log_psd_p10',
    'log_psd_range',
    'slope_log_psd',
    'flatness_psd',
    'n_picos_log',
    'densidade_picos',
    'std_sinal',
    'max_abs_sinal',
]

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df['target']
X_val = val_df[FEATURE_COLUMNS]
y_val = val_df['target']
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df['target']

print('Amostras por split:')
print(f'treino:   {len(train_ids)} sinais | {len(train_df)} janelas')
print(f'validacao:{len(val_ids)} sinais | {len(val_df)} janelas')
print(f'teste:    {len(test_ids)} sinais | {len(test_df)} janelas')

Amostras por split:
treino:   88 sinais | 4613 janelas
validacao:19 sinais | 1037 janelas
teste:    19 sinais | 1102 janelas


In [11]:
display(
    pd.DataFrame(
        {
            'split': ['treino', 'validacao', 'teste'],
            'janelas': [len(train_df), len(val_df), len(test_df)],
            'positivos': [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
            'taxa_positiva': [float(y_train.mean()), float(y_val.mean()), float(y_test.mean())],
        }
    )
)

,split,janelas,positivos,taxa_positiva
0,treino,4613,972,0.210709
1,validacao,1037,198,0.190935
2,teste,1102,255,0.231397


In [16]:
train_df.head()

,sample_id,start_hz,end_hz,center_hz,overlap_hz,overlap_fraction,target,energia_relativa_janela,log_psd_mean,log_psd_std,log_psd_max,log_psd_p90,log_psd_p10,log_psd_range,slope_log_psd,flatness_psd,n_picos_log,densidade_picos,std_sinal,max_abs_sinal
0,ef823432-8249-5cba-ace1-3c5192b825b7,1000,1250,1125.0,0.0,0.0,0,1.768891e-14,-169.200212,1.581182,-165.858344,-167.074692,-171.092721,9.540720,-0.007466,8.536772e-01,28,112.0,0.418864,2.381361
1,ef823432-8249-5cba-ace1-3c5192b825b7,1125,1375,1250.0,0.0,0.0,0,2.007080e-03,-166.607083,19.000560,-39.377299,-167.484148,-171.283394,136.021766,0.035332,1.485691e-11,29,116.0,0.418864,2.381361
2,ef823432-8249-5cba-ace1-3c5192b825b7,1250,1500,1375.0,0.0,0.0,0,2.007080e-03,-166.669721,19.007064,-39.377299,-167.794392,-171.617242,134.945790,-0.036838,1.462964e-11,28,112.0,0.418864,2.381361
3,ef823432-8249-5cba-ace1-3c5192b825b7,1375,1625,1500.0,0.0,0.0,0,1.556217e-14,-169.694361,1.539145,-166.378291,-167.948816,-171.653176,7.944797,-0.000365,8.480613e-01,28,112.0,0.418864,2.381361
4,ef823432-8249-5cba-ace1-3c5192b825b7,1500,1750,1625.0,0.0,0.0,0,1.616580e-14,-169.560850,1.601720,-165.267463,-167.642838,-171.568895,8.778267,0.003082,8.436373e-01,31,124.0,0.418864,2.381361


| Feature                   | O que captura                              |
| ------------------------- | ------------------------------------------ |
| `center_hz`               | Localização da janela no espectro          |
| `energia_relativa_janela` | Fração da energia concentrada na faixa     |
| `log_psd_mean`            | Nível médio de potência                    |
| `log_psd_std`             | Irregularidade interna do espectro         |
| `log_psd_max`             | Pico mais forte                            |
| `log_psd_p90`             | Nível superior robusto                     |
| `log_psd_p10`             | Piso espectral                             |
| `log_psd_range`           | Faixa dinâmica                             |
| `slope_log_psd`           | Tendência crescente ou decrescente         |
| `flatness_psd`            | Espectro tonal versus espalhado            |
| `n_picos_log`             | Complexidade e quantidade de componentes   |
| `densidade_picos`         | Picos por unidade de frequência            |
| `std_sinal`               | Intensidade global da vibração             |
| `max_abs_sinal`           | Maior impacto ou pico temporal             |
| `overlap_fraction`        | Construção do rótulo, não deve ser feature |
| `target`                  | Classe que o modelo deve prever            |


## Funções de Avaliação

O modelo irá retornar a probabilidade de ocorrência, logo a decisão final vai depender de um threshold.


In [12]:
def melhor_threshold_por_f1(y_true, y_prob):
    melhores = []
    for threshold in np.linspace(0.05, 0.95, 181):
        pred = (y_prob >= threshold).astype(int)
        melhores.append(
            {
                'threshold': float(threshold),
                'f1': f1_score(y_true, pred, zero_division=0),
            }
        )
    return max(melhores, key=lambda item: item['f1'])


def roc_auc_seguro(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)


def calcular_metricas(y_true, y_prob, threshold):
    pred = (y_prob >= threshold).astype(int)
    return {
        'accuracy': accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0),
        'roc_auc': roc_auc_seguro(y_true, y_prob),
    }

## Configuração dos experimentos que serão realizados

OBS: Poderiamos ter utilizado algum otimizador de Hyperparâmetros como Gridsearch ou optuna para conseguir produzir esse multiplos experimentos. Mas para fins de demonstração fiz diretamente pelo uma lista de configs por experimento.

In [13]:
pos_train = int(y_train.sum())
neg_train = int(len(y_train) - pos_train)
scale_pos_weight = neg_train / max(pos_train, 1)

configs = [
    {
        'nome': 'xgb_baseline',
        'n_estimators': 160,
        'max_depth': 3,
        'learning_rate': 0.05,
        'subsample': 0.90,
        'colsample_bytree': 0.90,
        'min_child_weight': 1,
        'reg_lambda': 1.0,
    },
    {
        'nome': 'xgb_multi_estimators',
        'n_estimators': 260,
        'max_depth': 3,
        'learning_rate': 0.05,
        'subsample': 0.90,
        'colsample_bytree': 0.90,
        'min_child_weight': 1,
        'reg_lambda': 1.0,
    },
    {
        'nome': 'xgb_depth4',
        'n_estimators': 160,
        'max_depth': 4,
        'learning_rate': 0.05,
        'subsample': 0.90,
        'colsample_bytree': 0.90,
        'min_child_weight': 1,
        'reg_lambda': 1.0,
    },
    {
        'nome': 'xgb_regularizado',
        'n_estimators': 220,
        'max_depth': 3,
        'learning_rate': 0.04,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'min_child_weight': 3,
        'reg_lambda': 2.0,
    },
    {
        'nome': 'xgb_depth4_modif',
        'n_estimators': 180,
        'max_depth': 4,
        'learning_rate': 0.04,
        'subsample': 0.90,
        'colsample_bytree': 0.90,
        'min_child_weight': 2,
        'reg_lambda': 1.5,
    },
]

resultados = []
modelos_treinados = {}

for config in configs:
    nome_run = config['nome']
    params_modelo = {
        **{k: v for k, v in config.items() if k != 'nome'},
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'scale_pos_weight': scale_pos_weight,
    }

    with mlflow.start_run(run_name=nome_run) as run:
        mlflow.set_tag('modelo_final_planejado', MODEL_NAME)
        mlflow.set_tag('tipo_modelagem', 'janela_frequencia_fft_psd')

        mlflow.log_params(params_modelo)
        mlflow.log_params(
            {
                'min_freq_hz': MIN_FREQ_HZ,
                'window_hz': WINDOW_HZ,
                'step_hz': STEP_HZ,
                'min_overlap_fraction': MIN_OVERLAP_FRACTION,
                'n_features': len(FEATURE_COLUMNS),
                'n_train_rows': len(train_df),
                'n_val_rows': len(val_df),
                'n_test_rows': len(test_df),
            }
        )

        modelo = XGBClassifier(**params_modelo)
        modelo.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

        val_prob = modelo.predict_proba(X_val)[:, 1]
        best = melhor_threshold_por_f1(y_val, val_prob)
        threshold = best['threshold']

        test_prob = modelo.predict_proba(X_test)[:, 1]
        metricas_val = calcular_metricas(y_val, val_prob, threshold)
        metricas_test = calcular_metricas(y_test, test_prob, threshold)

        for nome, valor in metricas_val.items():
            mlflow.log_metric(f'val_{nome}', valor)
        for nome, valor in metricas_test.items():
            mlflow.log_metric(f'test_{nome}', valor)
        mlflow.log_metric('threshold', threshold)

        importancias = (
            pd.DataFrame(
                {
                    'feature': FEATURE_COLUMNS,
                    'importance': modelo.feature_importances_,
                }
            )
            .sort_values('importance', ascending=False)
            .reset_index(drop=True)
        )

        predicoes_teste = test_df[['sample_id', 'start_hz', 'end_hz', 'target']].copy()
        predicoes_teste['prob_carpet'] = test_prob
        predicoes_teste['predicao'] = (test_prob >= threshold).astype(int)

        with tempfile.TemporaryDirectory() as tmpdir:
            tmpdir = Path(tmpdir)
            importancias_path = tmpdir / 'feature_importance.csv'
            predicoes_path = tmpdir / 'predicoes_teste.csv'
            features_path = tmpdir / 'feature_columns.json'
            config_path = tmpdir / 'config_features.json'

            importancias.to_csv(importancias_path, index=False)
            predicoes_teste.to_csv(predicoes_path, index=False)
            features_path.write_text(json.dumps(FEATURE_COLUMNS, indent=2), encoding='utf-8')
            config_path.write_text(
                json.dumps(
                    {
                        'min_freq_hz': MIN_FREQ_HZ,
                        'window_hz': WINDOW_HZ,
                        'step_hz': STEP_HZ,
                        'min_overlap_fraction': MIN_OVERLAP_FRACTION,
                    },
                    indent=2,
                ),
                encoding='utf-8',
            )

            mlflow.log_artifact(importancias_path, artifact_path='analise')
            mlflow.log_artifact(predicoes_path, artifact_path='predicoes')
            mlflow.log_artifact(features_path, artifact_path='config')
            mlflow.log_artifact(config_path, artifact_path='config')

        resultados.append(
            {
                'run_id': run.info.run_id,
                'nome': nome_run,
                'threshold': threshold,
                **{f'val_{k}': v for k, v in metricas_val.items()},
                **{f'test_{k}': v for k, v in metricas_test.items()},
            }
        )
        modelos_treinados[nome_run] = modelo

resultados_df = pd.DataFrame(resultados).sort_values(
    ['val_f1', 'val_recall', 'val_precision'],
    ascending=False,
)

In [14]:
display(resultados_df)

,run_id,nome,threshold,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
2,3c246f61f91e4073986b0b4a562690cc,xgb_depth4,0.760,0.945034,0.861538,0.848485,0.854962,0.975271,0.942831,0.936364,0.807843,0.867368,0.988351
1,a15eb3fd3c354b6f9ba4a75bb9acf802,xgb_multi_estimators,0.750,0.943105,0.845771,0.858586,0.852130,0.975954,0.942831,0.928571,0.815686,0.868476,0.988985
4,18bbe61b57c04428a7e56f47ec74e93e,xgb_depth4_modif,0.795,0.943105,0.863874,0.833333,0.848329,0.975277,0.944646,0.949074,0.803922,0.870488,0.988504
3,c8910268ae534bcfb99c0e934136e1f9,xgb_regularizado,0.800,0.941176,0.858639,0.828283,0.843188,0.974672,0.941924,0.944186,0.796078,0.863830,0.988073
0,0af7aaf9ba6a403d9d500f7ae9cc26ea,xgb_baseline,0.750,0.939248,0.839196,0.843434,0.841310,0.974630,0.941016,0.924107,0.811765,0.864301,0.987772


## Avaliação dos resultados via interface do MLFlow


```bash
mlflow ui --backend-store-uri ./mlruns
```

Depois abra o endereço mostrado pelo terminal, normalmente `http://127.0.0.1:5000`.

In [15]:
melhor_linha = resultados_df.iloc[0]
melhor_nome = melhor_linha['nome']
melhor_modelo = modelos_treinados[melhor_nome]
melhor_threshold = float(melhor_linha['threshold'])

test_prob = melhor_modelo.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= melhor_threshold).astype(int)

print(f'Melhor run: {melhor_nome}')
print(f'Threshold escolhido: {melhor_threshold:.3f}')
print('Matriz de confusao no teste:')
print(confusion_matrix(y_test, test_pred))
print('\nRelatorio no teste:')
print(classification_report(y_test, test_pred, zero_division=0))

pacote_modelo = {
    'model_name': MODEL_NAME,
    'modelo': melhor_modelo,
    'feature_columns': FEATURE_COLUMNS,
    'threshold': melhor_threshold,
    'feature_config': {
        'min_freq_hz': MIN_FREQ_HZ,
        'window_hz': WINDOW_HZ,
        'step_hz': STEP_HZ,
        'min_overlap_fraction': MIN_OVERLAP_FRACTION,
    },
    'best_run_name': melhor_nome,
    'best_run_id': melhor_linha['run_id'],
    'test_metrics': calcular_metricas(y_test, test_prob, melhor_threshold),
}

joblib.dump(pacote_modelo, MODEL_PKL_PATH)

with mlflow.start_run(run_name=f'{MODEL_NAME}_modelo_final') as final_run:
    mlflow.set_tag('modelo_final', MODEL_NAME)
    mlflow.set_tag('best_run_name', melhor_nome)
    mlflow.set_tag('best_run_id', melhor_linha['run_id'])

    mlflow.log_params(
        {
            'selected_model': melhor_nome,
            'threshold': melhor_threshold,
            'min_freq_hz': MIN_FREQ_HZ,
            'window_hz': WINDOW_HZ,
            'step_hz': STEP_HZ,
            'min_overlap_fraction': MIN_OVERLAP_FRACTION,
        }
    )
    for nome, valor in pacote_modelo['test_metrics'].items():
        mlflow.log_metric(f'test_{nome}', valor)

    mlflow.log_artifact(MODEL_PKL_PATH, artifact_path='modelo_pkl')

    try:
        mlflow.xgboost.log_model(
            melhor_modelo,
            artifact_path='modelo_xgboost',
            registered_model_name=MODEL_NAME,
        )
        print(f'Modelo registrado no MLflow como: {MODEL_NAME}')
    except Exception as exc:
        mlflow.set_tag('erro_registro_modelo', str(exc)[:500])
        mlflow.xgboost.log_model(melhor_modelo, artifact_path='modelo_xgboost')
        print('Modelo logado no MLflow, mas o registro por nome falhou.')
        print(str(exc))

print(f'Modelo salvo em: {MODEL_PKL_PATH}')
print(f'Run final MLflow: {final_run.info.run_id}')

Melhor run: xgb_depth4
Threshold escolhido: 0.760
Matriz de confusao no teste:
[[833  14]
 [ 49 206]]

Relatorio no teste:
              precision    recall  f1-score   support

           0       0.94      0.98      0.96       847
           1       0.94      0.81      0.87       255

    accuracy                           0.94      1102
   macro avg       0.94      0.90      0.92      1102
weighted avg       0.94      0.94      0.94      1102

Modelo registrado no MLflow como: Modelo_mlflow_v1s
Modelo salvo em: C:\Users\vinicius\Documents\MEUGITHUB\Vibration_challenge\models\part1\Modelo_mlflow_v1s.pkl
Run final MLflow: 9f9ae8d90db64118a4d26ce1b3bfeebc


Successfully registered model 'Modelo_mlflow_v1s'.
Created version '1' of model 'Modelo_mlflow_v1s'.
